In [ ]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

--2026-05-09 05:21:22--  https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 104.26.3.33, 104.26.2.33, 172.67.70.149, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|104.26.3.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26085508 (25M) [application/zip]
Saving to: ‘book-crossings.zip.1’

book-crossings.zip. 100%[===================>]  24.88M   125MB/s    in 0.2s    

2026-05-09 05:21:23 (125 MB/s) - ‘book-crossings.zip.1’ saved [26085508/26085508]

Archive:  book-crossings.zip
replace BX-Book-Ratings.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [ ]:
# add your code here - consider creating a new cell for each section of code

In [ ]:
def get_recommends(book = ""):
    book = book.strip()

    book_index = book_pivot.index.get_loc(book)

    distances, indices = model.kneighbors(
        [book_pivot.iloc[book_index]],
        n_neighbors=6
    )

    recommended_books = []

    for i in range(1, len(distances[0])):
        recommended_books.append([
            book_pivot.index[indices[0][i]],
            float(distances[0][i])
        ])

    return [book, recommended_books]

In [ ]:
df = df_ratings.merge(df_books, on='isbn')

# filter users with >= 200 ratings
user_counts = df['user'].value_counts()
df = df[df['user'].isin(user_counts[user_counts >= 200].index)]

# filter books with >= 100 ratings
book_counts = df['title'].value_counts()
df = df[df['title'].isin(book_counts[book_counts >= 100].index)]

# pivot table
book_pivot = df.pivot_table(
    index='title',
    columns='user',
    values='rating'
).fillna(0)

# model
model = NearestNeighbors(metric='cosine', algorithm='brute')
model.fit(book_pivot.values)